# Stage 1: UniMorph Data Fetching & Splitting
This notebook downloads raw UniMorph data for specified languages, performs train/val/test splits grouping by **lemma**, and saves them as compact Parquet files.

In [1]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/lora/data/raw-splits'
except ImportError:
    DATA_DIR =  '../data/raw-splits'

import os
os.makedirs(DATA_DIR, exist_ok=True)

In [2]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

LANGUAGES = ['eng', 'spa', 'swe']  # ISO 639-3 codes for UniMorph languages
UNIMORPH_BASE_URL = 'https://raw.githubusercontent.com/unimorph/{lang}/master/{lang}'

def fetch_and_split(lang, test_size=0.1, val_size=0.1, random_state=42):
    url = UNIMORPH_BASE_URL.format(lang=lang)
    print(f'Downloading {lang} data from {url}...')
    try:
        df = pd.read_csv(url, sep='\t', header=None, names=['lemma', 'target', 'features'], on_bad_lines='skip')
    except Exception as e:
        print(f'Error fetching {lang}: {e}')
        return
    
    df = df.dropna().reset_index(drop=True)
    
    # 1. First split: Separate out the test set by unique lemmas
    gss_test = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    train_val_idx, test_idx = next(gss_test.split(df, groups=df['lemma']))
    
    train_val_df = df.iloc[train_val_idx].reset_index(drop=True)
    test_df = df.iloc[test_idx].reset_index(drop=True)
    
    # 2. Second split: Separate train and validation sets from the remaining data
    val_relative_size = val_size / (1.0 - test_size)
    gss_val = GroupShuffleSplit(n_splits=1, test_size=val_relative_size, random_state=random_state)
    train_idx, val_idx = next(gss_val.split(train_val_df, groups=train_val_df['lemma']))
    
    train_df = train_val_df.iloc[train_idx].reset_index(drop=True)
    val_df = train_val_df.iloc[val_idx].reset_index(drop=True)
    
    # Save to parquet
    lang_dir = os.path.join(DATA_DIR, lang)
    os.makedirs(lang_dir, exist_ok=True)
    
    train_df.to_parquet(os.path.join(lang_dir, 'train.parquet'))
    val_df.to_parquet(os.path.join(lang_dir, 'val.parquet'))
    test_df.to_parquet(os.path.join(lang_dir, 'test.parquet'))
    
    print(f'Saved {lang}: train={len(train_df)}, val={len(val_df)}, test={len(test_df)}')
    print(f'Unique lemmas -> train={train_df["lemma"].nunique()}, val={val_df["lemma"].nunique()}, test={test_df["lemma"].nunique()}')

for lang in LANGUAGES:
    fetch_and_split(lang)

Saved eng: train=521740, val=65222, test=65506
Unique lemmas -> train=319656, val=39957, test=39957
Saved spa: train=953806, val=123415, test=119003
Unique lemmas -> train=52047, val=6506, test=6506
Saved swe: train=105154, val=13413, test=13130
Unique lemmas -> train=10016, val=1252, test=1252
